In [2]:
!pip install -q ttach

# Установка библиотек и Импорты

In [3]:
# Ячейка 1: Установка и Импорты
!pip install -q kagglehub timm albumentations

import kagglehub
import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tv_models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from huggingface_hub import hf_hub_download
import timm
import os
import json
import random
from collections import defaultdict
from glob import glob
from tqdm import tqdm

print("Библиотеки установлены и импортированы.")

Библиотеки установлены и импортированы.


In [4]:
# НОВАЯ ЯЧЕЙКА: Подключение Google Диска
from google.colab import drive
import os

# Подключаем диск (потребуется дать разрешение)
drive.mount('/content/drive')

# Создаем папку на вашем Google Диске для сохранения моделей
SAVE_DIR = '/content/drive/MyDrive/Colab_Models/Fruit_Classification/'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Все модели будут надежно сохраняться в: {SAVE_DIR}")

Mounted at /content/drive
Все модели будут надежно сохраняться в: /content/drive/MyDrive/Colab_Models/Fruit_Classification/


# Блок 2: Настройка окружения и Константы

In [5]:
# Ячейка 2: Конфигурация

# Фиксируем Random Seed
SEED = 9999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Определяем устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Словарь классов
class_to_idx = { "Апельсин": 0, "Бананы": 1, "Груши": 2, "Кабачки": 3, "Капуста": 4,
                 "Картофель": 5, "Киви": 6, "Лимон": 7, "Лук": 8, "Мандарины": 9,
                 "Морковь": 10, "Огурцы": 11, "Томаты": 12, "Яблоки зелёные": 13, "Яблоки красные": 14 }

Device: cpu


# Блок 3: Загрузка данных (Kaggle)

In [7]:
# Ячейка 3: Скачивание данных
kagglehub.login() # Может потребовать ввода токена

dl_lab_1_image_classification_path = kagglehub.competition_download('dl-lab-1-image-classification')
print(f'Data source import complete. Path: {dl_lab_1_image_classification_path}')

train_path_full = os.path.join(dl_lab_1_image_classification_path, 'train', 'train')
test_images_dir = os.path.join(dl_lab_1_image_classification_path, 'test_images', 'test_images')
sample_sub_path = os.path.join(dl_lab_1_image_classification_path, 'sample_submission.csv')

Kaggle credentials successfully validated.


100%|██████████| 194M/194M [00:01<00:00, 128MB/s]

Extracting files...


Data source import complete. Path: /root/.cache/kagglehub/competitions/dl-lab-1-image-classification


# Блок 4: Подготовка Датасета и Аугментации


In [8]:
# Ячейка 4: Классы Датасета и Аугментации
from albumentations.pytorch import ToTensorV2

train_transforms = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), p=0.5),

    # ИСПРАВЛЕНИЕ 1: Убрал always_apply (Resize и так применяется всегда)
    A.Resize(height=224, width=224),

    # 2. Геометрия
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    # ИСПРАВЛЕНИЕ 2: Заменили устаревший ShiftScaleRotate на современный Affine
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-180, 180), p=0.5),

    # 3. Эффекты
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=(3, 7), p=1.0),
        # ИСПРАВЛЕНИЕ 3: Вернули правильное название GaussNoise (без "ian")
        A.GaussNoise(p=1.0),
    ], p=0.2),

    # 4. Свет и Цвет
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),

    # 5. Блики и перекрытия
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 16), hole_width_range=(1, 16), fill_value=0, p=0.2),

    # 6. Финал
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    # ИСПРАВЛЕНИЕ 1: Убрали always_apply
    A.Resize(height=224, width=224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Класс Датасета
class FruitDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_filepath = row['filepath']
        label = row['label_idx']

        # Читаем
        image = cv2.imdecode(np.fromfile(image_filepath, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
        try:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        except Exception as e:
            # Заглушка для битых фото
            image = np.zeros((224, 224, 3), dtype=np.uint8)

        if self.transform is not None:
            image = self.transform(image=image)['image']

        return image, label

# Функция создания DataFrame
def create_dataframe(root_path):
    data = []
    all_images = glob(os.path.join(root_path, '*/*/*.jpg')) + \
                 glob(os.path.join(root_path, '*/*/*.png')) + \
                 glob(os.path.join(root_path, '*/*/*.jpeg'))

    for img_path in tqdm(all_images, desc="Parsing images"):
        norm_path = os.path.normpath(img_path)
        parts = norm_path.split(os.sep)

        plu_id = parts[-2]
        class_name = parts[-3]

        if class_name in class_to_idx:
            data.append({
                'filepath': img_path,
                'label_name': class_name,
                'label_idx': class_to_idx[class_name],
                'stratify_group': f"{class_name}_{plu_id}"
            })

    return pd.DataFrame(data)

/tmp/ipython-input-327/3101193192.py:30: UserWarning: Argument(s) 'fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 16), hole_width_range=(1, 16), fill_value=0, p=0.2),


# Блок 5: Парсинг данных (Создание таблицы)

In [9]:
# Ячейка 5: Создание DataFrame
df = create_dataframe(train_path_full)
print(f"Всего изображений: {len(df)}")
df.head()

Parsing images: 100%|██████████| 9889/9889 [00:00<00:00, 404378.16it/s]

Всего изображений: 9889


,filepath,label_name,label_idx,stratify_group
0,/root/.cache/kagglehub/competitions/dl-lab-1-i...,Лук,8,Лук_18387
1,/root/.cache/kagglehub/competitions/dl-lab-1-i...,Лук,8,Лук_18387
2,/root/.cache/kagglehub/competitions/dl-lab-1-i...,Лук,8,Лук_18387
3,/root/.cache/kagglehub/competitions/dl-lab-1-i...,Лук,8,Лук_18387
4,/root/.cache/kagglehub/competitions/dl-lab-1-i...,Лук,8,Лук_18387


# 5.1 Статистика по классам

In [10]:
# Ячейка 5.1: Статистика по классам

class_counts = df['label_idx'].value_counts().sort_index()
print("Количество изображений по классам (label_idx):")
print(class_counts)


Количество изображений по классам (label_idx):
label_idx
0     872
1     804
2     410
3     298
4     836
5     789
6     188
7     683
8     665
9     788
10    623
11    595
12    764
13    849
14    725
Name: count, dtype: int64


In [11]:
# Ячейка 5.2: функция лёгкого oversampling редких классов

def oversample_minority(train_df, min_count=500):
    """
    Если в классе меньше min_count картинок — дублируем их с заменой.
    Это делается ТОЛЬКО на train (не на val!).
    """
    counts = train_df['label_idx'].value_counts()
    dfs = [train_df]
    for cls, cnt in counts.items():
        if cnt < min_count:
            need = min_count - cnt
            extra = train_df[train_df['label_idx'] == cls].sample(
                need, replace=True, random_state=SEED
            )
            dfs.append(extra)
            print(f"Класс {cls}: было {cnt}, добавили {need}, стало {cnt + need}")
    return pd.concat(dfs).reset_index(drop=True)


# Блок 6: Логика Модели (Model Zoo)

In [12]:
# Ячейка 6: Загрузка модели с Grocery-весами (упрощённо)
def get_model(device):
    model_name = 'tf_efficientnet_b4_ns'
    print(f"Создаем модель: {model_name}")

    # Создаём модель под 15 классов с ImageNet-весами
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=15,
        drop_path_rate=0.3
    )

    # Загружаем наши Grocery-веса (только совпадающие ключи)
    pretrained_path = '/content/drive/MyDrive/Colab_Models/Grocery_Pretrain/grocery_b4_pretrain.pth'
    if os.path.exists(pretrained_path):
        print(f"📥 Загружаем Grocery-веса из: {pretrained_path}")
        checkpoint = torch.load(pretrained_path, map_location='cpu')

        model_state = model.state_dict()
        filtered_state = {}
        for k, v in checkpoint.items():
            if k in model_state and v.shape == model_state[k].shape:
                filtered_state[k] = v

        model.load_state_dict(filtered_state, strict=False)
        print(f"✅ Загружено {len(filtered_state)} ключей из Grocery.")
    else:
        print("⚠️ Grocery-веса не найдены, используем ImageNet.")

    model.to(device)
    return model

In [24]:
def get_model_for_inference(device):
    model_name = 'tf_efficientnet_b4_ns'
    model = timm.create_model(
        model_name,
        pretrained=False,          # Не загружаем ImageNet веса
        num_classes=15,
        drop_path_rate=0.3
    )
    model.to(device)
    return model

# Блок 7: Функции Обучения (Engine)

In [13]:
# Ячейка 7: Функции обучения и валидации
from tqdm.notebook import tqdm
@torch.no_grad()
def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_preds = []

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = loss_fn(logits, y)
        total_loss += loss.item() * y.size(0)

        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)
    f1 = f1_score(all_targets, all_preds, average='macro')
    return f1, avg_loss


def run_fold(fold, train_df, val_df, device, save_dir, n_epochs=33):
    scaler = torch.cuda.amp.GradScaler()
    print(f"\n{'='*10} FOLD {fold} {'='*10}")

    train_ds = FruitDataset(train_df, transform=train_transforms)
    val_ds   = FruitDataset(val_df,   transform=val_transforms)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

    model = get_model(device)

    labels = train_df['label_idx'].values
    weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    class_weights = torch.tensor(weights, dtype=torch.float).to(device)

    loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_f1 = 0.0
    best_val_loss = float('inf')
    history = []  # <-- добавили список для истории

    for epoch in range(1, n_epochs + 1):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch}")
        start_time = time.time()

        for X, y in pbar:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss_avg = train_loss / len(train_loader)
        train_f1, _ = evaluate(model, train_loader, loss_fn, device)
        val_f1, val_loss = evaluate(model, val_loader, loss_fn, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        epoch_time = time.time() - start_time

        # Вывод метрик
        print(f"Ep {epoch} [{epoch_time:.1f}s]: LR {current_lr:.2e} | "
              f"Train Loss {train_loss_avg:.4f} | Train F1 {train_f1:.4f} | "
              f"Val Loss {val_loss:.4f} | Val F1 {val_f1:.4f}")

        # Сохраняем метрики в историю
        history.append({
            'epoch': epoch,
            'train_loss': train_loss_avg,
            'train_f1': train_f1,
            'val_loss': val_loss,
            'val_f1': val_f1,
            'lr': current_lr,
            'time': epoch_time
        })

        # Сохранение лучших моделей
        if val_f1 > best_f1:
            best_f1 = val_f1
            save_path = os.path.join(save_dir, f"model_fold_{fold}_best_f1.pth")
            torch.save(model.state_dict(), save_path)
            print(f"--> Saved best F1: {best_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_path = os.path.join(save_dir, f"model_fold_{fold}_best_loss.pth")
            torch.save(model.state_dict(), save_path)
            print(f"--> Saved best Loss: {best_val_loss:.4f}")

    return best_f1, history  # <-- возвращаем и историю

# Блок 8: Запуск обучения (Main Loop)

In [ ]:
import time
# Ячейка 8: Запуск Cross-Validation
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_scores = []
all_histories = []       # для хранения истории каждого фолда
val_dfs = []             # для хранения валидационных датафреймов

try:
    split_gen = skf.split(df, df['stratify_group'])
except:
    split_gen = skf.split(df, df['label_idx'])

for fold, (train_idx, val_idx) in enumerate(split_gen):
    train_fold = df.iloc[train_idx].reset_index(drop=True)
    val_fold   = df.iloc[val_idx].reset_index(drop=True)

    # лёгкий oversampling редких классов только в train
    train_fold = oversample_minority(train_fold, min_count=500)

    score, history = run_fold(fold, train_fold, val_fold, device, save_dir=SAVE_DIR, n_epochs=33)
    fold_scores.append(score)
    all_histories.append(history)
    val_dfs.append(val_fold)   # сохраняем валидационный датафрейм для этого фолда

print(f"Average F1 across folds: {np.mean(fold_scores):.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


Класс 10: было 499, добавили 1, стало 500
Класс 11: было 476, добавили 24, стало 500
Класс 2: было 328, добавили 172, стало 500
Класс 3: было 238, добавили 262, стало 500
Класс 6: было 151, добавили 349, стало 500

========== FOLD 0 ==========
Создаем модель: tf_efficientnet_b4_ns


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

📥 Загружаем Grocery-веса из: /content/drive/MyDrive/Colab_Models/Grocery_Pretrain/grocery_b4_pretrain.pth
✅ Загружено 704 ключей из Grocery.


...

Ep 2:   0%|          | 0/273 [00:00<?, ?it/s]

Ep 2 [178.0s]: LR 2.97e-04 | Train Loss 1.0179 | Train F1 0.9070 | Val Loss 0.8559 | Val F1 0.9070
--> Saved best F1: 0.9070
--> Saved best Loss: 0.8559


Ep 3:   0%|          | 0/273 [00:00<?, ?it/s]

Ep 3 [177.8s]: LR 2.94e-04 | Train Loss 0.9228 | Train F1 0.9316 | Val Loss 0.8357 | Val F1 0.9112
--> Saved best F1: 0.9112
--> Saved best Loss: 0.8357


Ep 4:   0%|          | 0/273 [00:00<?, ?it/s]

In [14]:
import matplotlib.pyplot as plt
import numpy as np
# Если нет sklearn.metrics, тоже импортируем
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

for fold, hist in enumerate(all_histories):
    epochs = [h['epoch'] for h in hist]
    train_loss = [h['train_loss'] for h in hist]
    val_loss = [h['val_loss'] for h in hist]
    val_f1 = [h['val_f1'] for h in hist]
    lrs = [h['lr'] for h in hist]

    plt.figure(figsize=(15, 4))

    # График потерь
    plt.subplot(1, 3, 1)
    plt.plot(epochs, train_loss, label='Train Loss', marker='.')
    plt.plot(epochs, val_loss, label='Val Loss', marker='.')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title(f'Fold {fold} - Loss')
    plt.grid(True)

    # График F1
    plt.subplot(1, 3, 2)
    plt.plot(epochs, val_f1, label='Val F1', color='green', marker='.')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()
    plt.title(f'Fold {fold} - Validation F1')
    plt.grid(True)

    # График Learning Rate
    plt.subplot(1, 3, 3)
    plt.plot(epochs, lrs, label='Learning Rate', color='red', marker='.')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.legend()
    plt.title(f'Fold {fold} - Learning Rate')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

NameError: name 'all_histories' is not defined

# Inference (Предсказание)

In [22]:
import glob

# Путь к папке с моделями (укажите свой, если отличается)
SAVE_DIR = '/content/drive/MyDrive/Colab_Models/Fruit_Classification'

# Ищем все файлы модели по шаблону
model_files = glob.glob(os.path.join(SAVE_DIR, "model_fold_*_best_f1.pth"))
# Извлекаем номера фолдов
available_folds = sorted([int(f.split('_')[-3]) for f in model_files])
print(f"Найдены модели для фолдов: {available_folds}")

Найдены модели для фолдов: [0, 1, 2]


In [25]:
import ttach as tta

submission = pd.read_csv(sample_sub_path)
models = []

for fold in available_folds:
    path = os.path.join(SAVE_DIR, f"model_fold_{fold}_best_f1.pth")  # можно использовать best_loss или оба
    if os.path.exists(path):
        # Создаём модель без предзагрузки
        model = get_model_for_inference(device)
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()

        tta_model = tta.ClassificationTTAWrapper(model, tta.aliases.d4_transform())
        models.append(tta_model)
        print(f"✅ Модель {fold} загружена.")
    else:
        print(f"⚠️ Файл {path} не найден, пропускаем.")

if len(models) == 0:
    raise RuntimeError("Не загружено ни одной модели! Проверьте пути.")

predictions = []

with torch.no_grad():
    for img_id in tqdm(submission["image_id"], desc="Inference with TTA"):
        path = os.path.join(test_images_dir, img_id)
        img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_UNCHANGED)

        if img is None:
            # Если изображение не загрузилось, можно поставить 0 или пропустить
            predictions.append(0)
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = val_transforms(image=img)["image"].unsqueeze(0).to(device)

        fold_logits = []
        for model in models:
            logits = model(img_tensor)
            fold_logits.append(logits)

        avg_logits = torch.mean(torch.stack(fold_logits), dim=0)
        predictions.append(avg_logits.argmax(dim=1).item())

submission["label"] = predictions
submission.to_csv("submission_improved.csv", index=False)
print("✅ Файл submission_improved.csv готов!")

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


✅ Модель 0 загружена.
✅ Модель 1 загружена.
✅ Модель 2 загружена.


Inference with TTA:   0%|          | 0/2503 [00:00<?, ?it/s]

KeyboardInterrupt: 